# 🎯 LoRA Fine-Tuning: Introduction & Setup
## Visual LLM Educational Series - Module 1

Welcome to the **Visual LLM** hands-on LoRA fine-tuning series! This notebook will introduce you to LoRA (Low-Rank Adaptation) and set up your environment for practical fine-tuning.

### 📚 What You'll Learn:
- ✅ What is LoRA and why it's revolutionary
- ✅ How LoRA reduces memory requirements by 99%
- ✅ Setting up the environment for LoRA fine-tuning
- ✅ Installing required libraries (PEFT, transformers)
- ✅ Understanding the LoRA configuration

### ⏱️ Estimated Time: 15 minutes
### 🎯 Difficulty: Beginner
### 💻 Requirements: Free Google Colab (GPU recommended)

---

**🔗 Part of the Visual LLM Educational Platform**  
Visit: [Visual LLM Platform](https://your-visual-llm-app.herokuapp.com) for the complete curriculum!

## 🧠 Understanding LoRA (Low-Rank Adaptation)

### What is LoRA?
LoRA is a **parameter-efficient fine-tuning technique** that allows you to adapt large language models using only a tiny fraction of the original parameters.

### Key Benefits:
- 🔥 **99% fewer trainable parameters** (7B model: 7B → 4.2M parameters)
- 💾 **Massive memory savings** (28GB → 14GB for 7B model)
- ⚡ **Faster training** (hours instead of days)
- 💰 **Cost effective** (train on consumer GPUs)
- 🔄 **Easy to merge** and deploy

### How LoRA Works:
Instead of updating the full weight matrix **W**, LoRA decomposes the update into two smaller matrices:

```
W_new = W_original + A × B
```

Where:
- **A**: Matrix of size (d × r)
- **B**: Matrix of size (r × k)  
- **r**: Rank (typically 4-64, much smaller than d or k)

This reduces parameters from **d × k** to **d × r + r × k**!

## 🛠️ Environment Setup

Let's set up your environment for LoRA fine-tuning!

In [ ]:
# Check GPU availability
import torch
print(f"🔥 CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📱 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected. LoRA will still work but will be slower.")
    print("💡 Go to Runtime > Change runtime type > Hardware accelerator > GPU")

In [ ]:
# Install required packages for LoRA fine-tuning
print("📦 Installing LoRA fine-tuning packages...")

!pip install -q transformers==4.36.0
!pip install -q peft==0.7.1
!pip install -q datasets==2.14.0
!pip install -q accelerate==0.24.0
!pip install -q bitsandbytes==0.41.0
!pip install -q trl==0.7.4

print("✅ Installation complete!")
print("🎯 Ready for LoRA fine-tuning!")

In [ ]:
# Import essential libraries
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training
)
from datasets import Dataset
import json
import os

print("📚 Libraries imported successfully!")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers version: {transformers.__version__}")
print(f"⚡ PEFT version: {peft.__version__}")

## ⚙️ LoRA Configuration

Let's understand the key LoRA parameters:

In [ ]:
# LoRA Configuration - Understanding the parameters
lora_config = LoraConfig(
    r=16,                    # Rank: Higher = more parameters, better quality
    lora_alpha=32,           # Scaling factor: Usually 2x the rank
    target_modules=[         # Which layers to apply LoRA to
        "q_proj",           # Query projection
        "k_proj",           # Key projection  
        "v_proj",           # Value projection
        "o_proj",           # Output projection
        "gate_proj",        # Gate projection (for Llama)
        "up_proj",          # Up projection
        "down_proj",        # Down projection
    ],
    lora_dropout=0.1,        # Dropout for regularization
    bias="none",             # Don't adapt bias parameters
    task_type=TaskType.CAUSAL_LM,  # Causal language modeling
)

print("⚙️ LoRA Configuration:")
print(f"   📊 Rank (r): {lora_config.r}")
print(f"   📈 Alpha: {lora_config.lora_alpha}")
print(f"   🎯 Target modules: {len(lora_config.target_modules)}")
print(f"   🔄 Dropout: {lora_config.lora_dropout}")
print(f"   📝 Task type: {lora_config.task_type}")

### 📊 Parameter Explanation:

**Rank (r):**
- Controls the "capacity" of LoRA adaptation
- Higher rank = more parameters = better quality (but more memory)
- Typical values: 4, 8, 16, 32, 64
- Sweet spot: 16-32 for most tasks

**Alpha:**
- Scaling factor for LoRA updates
- Usually set to 2x the rank
- Controls how much the LoRA adaptation affects the model

**Target Modules:**
- Which layers to apply LoRA to
- More modules = better adaptation (but more parameters)
- Common choices: attention layers (q_proj, v_proj) or all linear layers

In [ ]:
# Calculate parameter reduction with LoRA
def calculate_lora_parameters(model_size_b, rank, num_target_modules, hidden_size=4096):
    """
    Calculate LoRA parameter reduction
    """
    # Original model parameters
    original_params = model_size_b * 1e9
    
    # LoRA parameters per module: (hidden_size * rank) + (rank * hidden_size)
    lora_params_per_module = 2 * hidden_size * rank
    total_lora_params = lora_params_per_module * num_target_modules
    
    # Calculate reduction
    reduction_factor = original_params / total_lora_params
    percentage_reduction = (1 - total_lora_params / original_params) * 100
    
    return {
        'original_params': original_params,
        'lora_params': total_lora_params,
        'reduction_factor': reduction_factor,
        'percentage_reduction': percentage_reduction
    }

# Example calculations for different model sizes
models = [
    ("Llama-2-7B", 7, 7),
    ("Llama-2-13B", 13, 7), 
    ("Llama-2-70B", 70, 7)
]

print("📊 LoRA Parameter Reduction Analysis:")
print("=" * 60)

for model_name, size_b, num_modules in models:
    stats = calculate_lora_parameters(size_b, 16, num_modules)
    
    print(f"\n🤖 {model_name}:")
    print(f"   📈 Original: {stats['original_params']/1e9:.1f}B parameters")
    print(f"   ⚡ LoRA: {stats['lora_params']/1e6:.1f}M parameters")
    print(f"   🎯 Reduction: {stats['reduction_factor']:.0f}x fewer parameters")
    print(f"   💾 Memory saved: {stats['percentage_reduction']:.1f}%")

## 🎯 Next Steps

Congratulations! You've successfully:
- ✅ Understood what LoRA is and why it's powerful
- ✅ Set up your environment for LoRA fine-tuning
- ✅ Configured LoRA parameters
- ✅ Calculated the massive parameter reduction benefits

### 📚 Continue Your Learning:

**Next Notebook:** [02_LoRA_Practical_Fine_Tuning.ipynb](link-to-next-notebook)
- Load a pre-trained model
- Apply LoRA configuration
- Fine-tune on a custom dataset
- Save and load LoRA adapters

**Visual LLM Platform:** [Complete Curriculum](https://your-visual-llm-app.herokuapp.com)
- Interactive workshops
- AI-powered learning assistant
- Comprehensive tutorials
- Community support

### 💡 Key Takeaways:
1. **LoRA reduces parameters by 99%** while maintaining quality
2. **Rank controls the adaptation capacity** (sweet spot: 16-32)
3. **Target modules determine which layers adapt** (more = better)
4. **LoRA makes fine-tuning accessible** on consumer hardware

---

**🎓 Ready to fine-tune your first model with LoRA?**  
Continue to the next notebook for hands-on practice!

In [ ]:
# Save your progress (optional)
progress = {
    "module": "01_LoRA_Introduction",
    "completed": True,
    "timestamp": str(pd.Timestamp.now()),
    "gpu_used": torch.cuda.is_available(),
    "lora_config": {
        "rank": lora_config.r,
        "alpha": lora_config.lora_alpha,
        "target_modules": len(lora_config.target_modules)
    }
}

# Save to file
with open('lora_progress.json', 'w') as f:
    json.dump(progress, f, indent=2)

print("💾 Progress saved!")
print("🎯 Ready for the next module!")